In [ ]:
!pip install transformers datasets librosa torchaudio accelerate matplotlib captum

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset, Audio, DatasetDict
from huggingface_hub import create_branch
from scipy.special import softmax
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, cohen_kappa_score, roc_auc_score,
    log_loss, average_precision_score,
)
from sklearn.preprocessing import label_binarize
from transformers import (
    AutoFeatureExtractor, AutoModelForAudioClassification,
    EarlyStoppingCallback, Trainer, TrainingArguments, set_seed,
)

#Δημιουργία των branches στο repo, για κάθε seed

In [ ]:
from huggingface_hub import HfApi, create_branch

repo_id = "VasilisAsim/data2vec-finetuned-TESS_full"
api = HfApi()

seeds = [5, 6]
for seed in seeds:
    branch_name = f"seed-{seed}"
    try:
        create_branch(repo_id=repo_id, branch=branch_name, repo_type="model")
        print(f"Δημιουργήθηκε branch: {branch_name}")
    except Exception as e:
        print(f"Branch {branch_name} υπάρχει ήδη ή σφάλμα: {e}")

In [ ]:
def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))   # για το filter

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")


    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})

In [ ]:
dataset = load_tess_speaker_independent()

In [ ]:
#Εξαγωγή των ετικετών των συναισθημάτων σε λίστα
labels = sorted(set(dataset["train"]["emotion"]))
print(labels)

#Δημιουργία λεξικών αντιστοίχισης ετικετών και αριθμών
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = i
    id2label[i] = label

#υπολογισμός αριθμού ετικετών
num_labels = len(labels)
print(f"Διαθέσιμες ετικέτες: {labels}")
print(f"Αριθμός ετικετών: {num_labels}")

In [ ]:
# Φόρτωση του feature extractor του βασικού μοντέλου
feature_extractor = AutoFeatureExtractor.from_pretrained("facebook/data2vec-audio-base-960h")

# Συνάρτηση για την προετοιμασία του dataset
def prepare_dataset(batch):
    audio = batch["audio"]
    #Μετατροπή δείγματος ομιλίας σε κατάλληλη μορφή για το finetuning
    batch["input_values"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    #Μεταονομασία της μεταβλητής για συμβατότητα με τον trainer
    batch["labels"] = batch["emotion"]
    return batch

#Εφαρμογή της συνάρτησης σε όλο το dataset και αφαίρεση των υπόλοιπων στηλών
encoded_dataset = dataset.map(prepare_dataset, remove_columns=dataset["train"].column_names)

#Εκτύπωση της δομής του νεου dataset
print("Encoded dataset features:")
print(encoded_dataset["train"].features)

#Εκτύπωση ενός δείγματος
print("\nSample from encoded dataset:")
print(encoded_dataset["train"][0])

### Ορισμός μετρικών αξιολόγησης

In [ ]:
#Συνάρτηση υπολογισμού των μετρικών αξιολόγησης
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probs = softmax(logits, axis=1)

    num_classes = probs.shape[1]
    labels_bin = label_binarize(labels, classes=list(range(num_classes)))

    # ROC-AUC (macro-averaged, one-vs-rest)
    roc_auc = roc_auc_score(
        labels_bin, probs, multi_class="ovr", average="macro"
    )

    # PR-AUC (macro-averaged average precision)
    pr_auc = average_precision_score(
        labels_bin, probs, average="macro"
    )

    accuracy = accuracy_score(labels, predictions)
    balanced_accuracy = balanced_accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average="weighted", zero_division=0)
    recall = recall_score(labels, predictions, average="weighted", zero_division=0)
    f1 = f1_score(labels, predictions, average="weighted", zero_division=0)
    mcc = matthews_corrcoef(labels, predictions)
    k_score = cohen_kappa_score(labels, predictions)
    logloss = log_loss(labels, probs)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mcc" : mcc,
        "k_score" : k_score,
        "log_loss": logloss,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc
    }

### Φόρτωση του αρχικού προεκπαιδευμένου μοντέλου

In [ ]:
label2id_fixed = {str(label): i for i, label in enumerate(labels)}
id2label_fixed = {i: str(label) for i, label in enumerate(labels)}

### Ορισμός παραμέτρων εκπαίδευσης

In [ ]:
#Δημιουργία DataCollator για padding των δεδομένων
class DataCollatorCTCWithPadding:
    def __init__(self, feature_extractor, padding=True):
        self.feature_extractor = feature_extractor
        self.padding = padding

    def __call__(self, features):
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        batch = self.feature_extractor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )
        # Convert string labels to integer IDs using the global label2id dictionary
        batch["labels"] = torch.tensor([label2id[feature["labels"]] for feature in features], dtype=torch.long)
        return batch

data_collator = DataCollatorCTCWithPadding(feature_extractor=feature_extractor, padding=True)

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, login

login(token=userdata.get("HF_TOKEN"))

api = HfApi()
repo_id = "VasilisAsim/data2vec-finetuned-TESS_full"

for seed in [3, 4]:
    api.delete_branch(repo_id=repo_id, branch=f"seed-{seed}")
    print(f"Διαγράφηκε το branch seed-{seed}")

print([b.name for b in api.list_repo_refs(repo_id).branches])

In [ ]:
SEEDS = [3, 4, 7]          # μόνο το 6· το 5 έχει ήδη ανέβει
results = {}

for seed in SEEDS:
    print(f"\n{'='*50}\nΕκπαίδευση με seed={seed}\n{'='*50}")
    set_seed(seed)

    model = AutoModelForAudioClassification.from_pretrained(
        "facebook/data2vec-audio-base-960h",
        num_labels=num_labels,
        label2id=label2id_fixed,
        id2label=id2label_fixed,
        ignore_mismatched_sizes=True,
    )
    model.freeze_feature_encoder()

    training_args = TrainingArguments(
        output_dir=f"./results_seed_{seed}",
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=4,
        eval_strategy="epoch",
        save_strategy="epoch",
        num_train_epochs=8,
        fp16=True,
        logging_steps=50,
        learning_rate=3e-5,
        warmup_steps=500,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        seed=seed,
        push_to_hub=False,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=encoded_dataset["train"],
        eval_dataset=encoded_dataset["test"],
        compute_metrics=compute_metrics,
        data_collator=data_collator,
    )

    trainer.train()
    print("max_steps:", trainer.state.max_steps)

    results[seed] = trainer.evaluate()
    print(results[seed])

    # Push στο branch του seed (όχι μέσω Trainer)
    branch_name = f"seed-{seed}"
    model.save_pretrained(f"./seed{seed}_best")      # τοπικό αντίγραφο ασφαλείας
    model.push_to_hub(repo_id, revision=branch_name, commit_message=f"Fine-tuned with seed={seed}")
    feature_extractor.push_to_hub(repo_id, revision=branch_name)
    print(f"Ολοκληρώθηκε και ανέβηκε: {branch_name}")

    del model, trainer
    torch.cuda.empty_cache()

### Εξέλιξη του accuracy ανά epoch για διαφορετικά seeds

In [ ]:
acc = {
    0: [0.1786, 0.1486, 0.1579, 0.1900, 0.4114, 0.5514, 0.6186, 0.6829],
    1: [0.1500, 0.1743, 0.1429, 0.1843, 0.1864, 0.1886, 0.5564, 0.6871],
    2: [0.1629, 0.2150, 0.1614, 0.2507, 0.3264, 0.3793, 0.4014, 0.6771],
    3: [0.1564, 0.2157, 0.1700, 0.4407, 0.4579, 0.5300, 0.5750, 0.5836],
    4: [0.1750, 0.1493, 0.1950, 0.1521, 0.3993, 0.4636, 0.5214, 0.5243],
    5: [0.1428, 0.1657, 0.1657, 0.2050, 0.3028, 0.3342, 0.4492, 0.4657],
    6: [0.1421, 0.1500, 0.2385, 0.3678, 0.3685, 0.4071, 0.4821, 0.6000],
    7: [0.1336, 0.1736, 0.1507, 0.3493, 0.4221, 0.4194, 0.4436, 0.5221],
}
epochs = np.arange(1, 9)
mat = np.array(list(acc.values()))

fig, ax = plt.subplots(figsize=(8, 5))
for seed, a in acc.items():
    ax.plot(epochs, a, "-o", alpha=0.45, label=f"seed {seed}")
ax.plot(epochs, mat.mean(0), "k-s", lw=2.5, label="μέσος όρος")
ax.fill_between(epochs, mat.mean(0) - mat.std(0, ddof=1),
                mat.mean(0) + mat.std(0, ddof=1), color="k", alpha=0.12)
ax.axhline(1 / 7, color="gray", ls="--", label="τύχη (1/7)")
ax.axhline(0.741, color="red", ls=":", label="baseline (seed 42)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy (σύνολο δοκιμής)")
ax.set_title("Εξέλιξη του accuracy ανά epoch για διαφορετικά seeds")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

final = np.array([0.683, 0.687, 0.677, 0.584, 0.524, 0.466, 0.600, 0.522])
print(f"mean={final.mean():.3f} std={final.std(ddof=1):.3f} "
      f"min={final.min():.3f} max={final.max():.3f}")